# Learning Rate and Batch Size Trial

### Revision guide

This notebook compares the listed learning-rate, decay and mini-batch settings for a NumPy **multilayer perceptron (MLP)** that attributes clips to five music generators. Inputs are the configured Fake Music Caps WAV folders/ZIP or shared feature caches. Each clip becomes 80 aggregated mel-frequency cepstral coefficient (MFCC) features. Training fits the network; validation guides early stopping and configuration ranking; test audio is not evaluated here.

Outputs are validation-score and best-variant history CSV tables, a learning-curve image and a JSON summary under this trial's `task1_outputs/numpy_mlp_trials` folder, plus feature caches when needed. Weights are not saved here. `00_MLP_Baseline.ipynb` supplies the reference; `06_MLP_Trial_Comparison.ipynb` reads this trial's validation CSV alongside the others and passes selected settings to `07_MLP_Final_Selected_Model.ipynb`.


## Goal

Compare learning-rate, decay, and batch-size settings while keeping the selected baseline architecture, activation, dropout, and optimizer family fixed.

## Setup

The configuration cell is the only cell normally edited in Colab.

In [1]:
# Install only missing packages. Neural-network notebooks do not install or use
# TensorFlow, PyTorch, Keras, or scikit-learn neural-network implementations.
# import makes a module available; dotted names access a part of that module.
# These standard-library tools check packages and can launch the current Python.
import importlib.util
import subprocess
import sys

# = assigns a value to a name. This dictionary maps import names (keys) to
# package-installation names (values); scikit-learn imports as sklearn.
required_packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
    "joblib": "joblib",
}
# The list comprehension loops over dictionary items, unpacking each key/value
# pair. It keeps the package name only if find_spec returns None (not found).
missing = [pip_name for module, pip_name in required_packages.items()
           if importlib.util.find_spec(module) is None]
# A non-empty list makes this condition true. check_call runs pip using this
# Python executable; *missing expands the list into separate command arguments.
# This cell can install packages when run; it does not load any data partition.
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("Environment ready.")

Environment ready.


In [2]:
# These imports provide file, timing, hashing and other utilities. Importing a
# module does not call its functions; some imported helpers are unused here.
import copy
import hashlib
import json
import math
import os
import random
import time
import warnings
import zipfile
from pathlib import Path

# as gives a shorter alias: np for NumPy arrays, pd for pandas tables and plt
# for plotting. from ... import selects a particular object from a module.
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.io import wavfile
from scipy.signal import resample_poly

# A seed makes these random sequences repeatable. The MLP later creates its
# own seeded Generator object rather than relying only on these global seeds.
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

# and short-circuits: check google.colab only if google is available. The
# if/else branches choose Google Colab Drive paths or a local working folder.
IN_COLAB = (
    importlib.util.find_spec("google") is not None
    and importlib.util.find_spec("google.colab") is not None
)
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    default_root = Path("/content/drive/MyDrive/Colab Notebooks/Education/ProgMathsAI")
else:
    default_root = Path.cwd()
    if not (default_root / "Fake Music Caps.zip").exists() and (default_root.parent / "Fake Music Caps.zip").exists():
        default_root = default_root.parent

# These environment variables provide a local-test override without changing
# the Colab defaults shown to a coursework reader.
# Path creates a filesystem-path object. os.environ.get(name, default) reads
# an environment variable or uses its fallback; str converts a Path to text.
PROJECT_ROOT = Path(os.environ.get("PROGMATHSAI_PROJECT_ROOT", str(default_root)))
ZIP_PATH = Path(os.environ.get("FAKEMUSICCAPS_ZIP_PATH", str(PROJECT_ROOT / "Datasets" / "Fake Music Caps.zip")))
DATA_DIR = Path(os.environ.get("FAKEMUSICCAPS_DATA_DIR", str(PROJECT_ROOT / "Datasets" / "Fake Music Caps")))
OUTPUT_ROOT = Path(os.environ.get("FAKEMUSICCAPS_OUTPUT_ROOT", str(PROJECT_ROOT / "task1_outputs")))

# Unlike the analysis notebooks, this notebook fixes final mode directly;
# the smoke environment setting does not change its sample or epoch limits.
RUN_MODE = "final"
# Final submission mode is fixed intentionally.
# These notebooks always use the full configured split, full epoch/patience settings,
# and _final output files.

# For Path objects, / joins folders. TRIAL_TAG gives this experiment its own
# output folder, while CACHE_DIR is shared by the MLP notebooks.
TRIAL_TAG = "05_learningrate_batchsize_trial"
MODEL_TAG = "numpy_mlp_trials"
OUTPUT_DIR = OUTPUT_ROOT / MODEL_TAG / TRIAL_TAG
FIGURES_DIR = OUTPUT_DIR / "figures"
METRICS_DIR = OUTPUT_DIR / "metrics"
MODELS_DIR = OUTPUT_DIR / "models"
TABLES_DIR = OUTPUT_DIR / "tables"
CACHE_DIR = OUTPUT_ROOT / MODEL_TAG / "cache"
# for visits each Path in the tuple. mkdir is a method on that object;
# parents=True creates missing parents, and exist_ok=True accepts existing folders.
for directory in (FIGURES_DIR, METRICS_DIR, MODELS_DIR, TABLES_DIR, CACHE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# Map each generator folder to a numeric target. List indexing starts at zero,
# so CLASS_NAMES[label] recovers the matching readable class name.
FOLDER_TO_LABEL = {
    "audioldm2": 0,
    "MusicGen_medium": 1,
    "musicldm": 2,
    "mustango": 3,
    "stable_audio_open": 4,
}
CLASS_NAMES = ["AudioLDM2", "MusicGen", "MusicLDM", "Mustango", "Stable Audio Open"]

# Audio settings: 16,000 samples/second and 2.5 seconds give 40,000 samples.
# An underscore in a number is only a readability aid. MS settings are milliseconds.
# FFT = fast Fourier transform; MFCC = mel-frequency cepstral coefficient.
SAMPLE_RATE = 16_000
CLIP_SECONDS = 2.5
FRAME_MS = 25
HOP_MS = 10
N_FFT = 512
N_MELS = 26
N_MFCC = 20
POOL_FACTOR = 4

# Display the chosen paths/settings for inspection; no dataset is read here.
print({
    "run_mode": RUN_MODE,
    "trial_tag": TRIAL_TAG,
    "project_root": str(PROJECT_ROOT),
    "zip_path": str(ZIP_PATH),
    "data_dir": str(DATA_DIR),
    "output_dir": str(OUTPUT_DIR),
})

ValueError: mount failed

## Data

Track IDs, rather than individual generated files, determine the split. Each split is balanced independently.

### Which partition is used?

**Training** rows fit feature scaling and network weights. **Validation** rows use those same scaling statistics and guide early stopping and reported trial scores. **Test** rows are listed and balanced in the manifest (the file inventory), but their audio is not loaded for features or evaluation here.

All generator files sharing a track ID receive the same split. This avoids putting different generated versions of one track in both training and validation/test. Balancing samples each partition down to its smallest class; it can remove individual file rows without changing their split assignment.


In [3]:
# def defines a function; its indented body runs only when the function is
# called with (). This function takes no arguments and returns a dataset Path.
def ensure_dataset_available():
    """Extract the archive only when the configured data directory is absent."""
    # exists() returns a Boolean. return immediately exits this call, so an
    # existing dataset folder bypasses archive extraction.
    if DATA_DIR.exists():
        return DATA_DIR
    # not reverses the condition. raise stops execution with an explanatory error;
    # f-strings insert values from braces into the message.
    if not ZIP_PATH.exists():
        raise FileNotFoundError(
            f"Dataset folder not found at {DATA_DIR} and ZIP not found at {ZIP_PATH}. "
            "Update the configuration cell or upload the ZIP to Google Drive."
        )
    print(f"Extracting {ZIP_PATH.name}; this is a one-time operation...")
    # with closes the archive automatically after this block. extractall writes
    # files below DATA_DIR.parent; the next check verifies the expected folder.
    with zipfile.ZipFile(ZIP_PATH) as archive:
        archive.extractall(DATA_DIR.parent)
    if not DATA_DIR.exists():
        raise FileNotFoundError(f"Archive extracted, but expected folder was not created: {DATA_DIR}")
    return DATA_DIR


# Calling the helper below finds/extracts the dataset. This scan reads folder
# and filename information for all partitions, not the WAV audio samples yet.
def scan_dataset():
    root = ensure_dataset_available()
    missing_folders = [folder for folder in FOLDER_TO_LABEL if not (root / folder).is_dir()]
    if missing_folders:
        raise FileNotFoundError(f"Missing expected generator folders: {missing_folders}")

    # Build one dictionary per WAV file. The outer loop unpacks folder/label pairs;
    # the inner loop visits sorted file paths. path.stem is the name without .wav.
    rows = []
    for folder, label in FOLDER_TO_LABEL.items():
        paths = sorted((root / folder).glob("*.wav"))
        if not paths:
            raise ValueError(f"No WAV files found in {root / folder}")
        for path in paths:
            rows.append({
                "path": str(path),
                "track_id": path.stem,
                "folder": folder,
                "label": label,
                "class_name": CLASS_NAMES[label],
            })

    # DataFrame turns the row dictionaries into a table. ["path"] selects a column;
    # duplicated().any() asks whether any path appears more than once.
    manifest = pd.DataFrame(rows)
    if manifest["path"].duplicated().any():
        raise AssertionError("Duplicate file paths found in the manifest.")
    # The r prefix makes a raw string for the regular expression: exactly 11 allowed
    # characters. .str applies string operations to the whole pandas column.
    valid_track_ids = manifest["track_id"].str.fullmatch(r"[A-Za-z0-9_-]{11}")
    if not valid_track_ids.all():
        # .loc[row_mask, column] selects matching rows and a column. ~ reverses this
        # Boolean mask; chained methods then show up to ten distinct invalid IDs.
        invalid = manifest.loc[~valid_track_ids, "track_id"].drop_duplicates().head(10).tolist()
        raise ValueError(
            "Expected every WAV stem to be an 11-character track ID containing only "
            f"letters, digits, '_' or '-'. Invalid examples: {invalid}"
        )
    # Return a consistently ordered table. reset_index(drop=True) gives it fresh
    # row numbers without keeping the old index as another column.
    return manifest.sort_values(["track_id", "label"]).reset_index(drop=True)


# track_id is an argument; seed has a default if the caller omits it. SHA-256
# (Secure Hash Algorithm, 256-bit) maps the seeded ID to a repeatable digest.
def split_for_track_id(track_id, seed=RANDOM_SEED):
    # Stable across Python sessions, operating systems, and notebook order.
    digest = hashlib.sha256(f"{seed}:{track_id}".encode("utf-8")).hexdigest()
    # [:8] takes the first eight characters (end excluded); int(..., 16) reads
    # hexadecimal. % takes a remainder, giving a bucket from 0 to 9,999.
    # Thresholds target 70%/15%/15%; actual split sizes need not match exactly.
    bucket = int(digest[:8], 16) % 10_000
    if bucket < 7_000:
        return "train"
    if bucket < 8_500:
        return "validation"
    return "test"


# Take a manifest table and return a selected, balanced table. copy() keeps
# the caller's table separate; map calls split_for_track_id for each track ID.
def make_balanced_splits(manifest):
    selected_parts = []
    working = manifest.copy()
    working["split"] = working["track_id"].map(split_for_track_id)

    # enumerate supplies both the position and split name. Boolean indexing keeps
    # only this partition; groupby/size counts its files per class. reindex includes
    # all five labels, using zero for any absent class.
    for split_index, split_name in enumerate(("train", "validation", "test")):
        split_df = working[working["split"] == split_name]
        counts = split_df.groupby("label").size().reindex(range(len(CLASS_NAMES)), fill_value=0)
        if (counts == 0).any():
            raise ValueError(f"At least one class is absent from {split_name}: {counts.to_dict()}")
        # Balance each partition to its own smallest class by sampling without
        # replacement. The seeded samples select rows; they do not reassign track IDs.
        per_class = int(counts.min())
        for label in range(len(CLASS_NAMES)):
            class_rows = split_df[split_df["label"] == label]
            selected_parts.append(class_rows.sample(
                n=per_class,
                random_state=RANDOM_SEED + split_index * 100 + label,
                replace=False,
            ))

    # concat joins the sampled tables vertically into one selected manifest.
    selected = pd.concat(selected_parts, ignore_index=True)
    selected = selected.sort_values(["split", "label", "track_id"]).reset_index(drop=True)

    # This dictionary comprehension builds a set of unique track IDs per split.
    # The assertions stop if any ID crosses partitions, even across generators.
    id_sets = {
        split: set(selected.loc[selected["split"] == split, "track_id"])
        for split in ("train", "validation", "test")
    }
    assert id_sets["train"].isdisjoint(id_sets["validation"])
    assert id_sets["train"].isdisjoint(id_sets["test"])
    assert id_sets["validation"].isdisjoint(id_sets["test"])
    return selected


# Return a count table grouped by split and generator for the printed audit.
def split_summary(selected):
    return (selected.groupby(["split", "class_name"])
            .size().rename("files").reset_index())

## NumPy MFCCs

Framing, FFT, mel filters, logarithms, and the DCT are implemented explicitly with NumPy.

### From audio to 80 features

**Mel-frequency cepstral coefficients (MFCCs)** summarise a clip's short-time spectral shape. The code windows the waveform into overlapping frames, applies a fast Fourier transform (FFT), combines power with mel-spaced filters, takes logarithms and applies a discrete cosine transform (DCT).

With these settings, one clip becomes **249 frames x 20 coefficients**. This experiment keeps the mean and standard deviation of each coefficient over time, plus the same statistics for its time derivative (delta): **20 + 20 + 20 + 20 = 80 features**. This aggregation removes explicit frame order from the MLP input. Pooling into sequences/maps is available in the helpers but is not this experiment's chosen representation.


In [4]:
# Given a WAV path, return a one-dimensional float32 waveform. Defaults use
# the configured sample rate and duration; no statistics are fitted to a split.
# float32 stores 32-bit floating-point values; float64 uses 64 bits.
def load_wav_center_crop(path, target_sr=SAMPLE_RATE, seconds=CLIP_SECONDS):
    """Load, mono-convert, resample, and deterministically centre crop/pad a WAV."""
    # read returns two values, unpacked into names: samples/second and an array.
    # .ndim and .dtype are array attributes; astype(...) and mean(...) are methods.
    # For stereo, axis=1 averages channels within each sample to produce mono.
    sample_rate, audio = wavfile.read(path)
    if audio.ndim == 2:
        audio = audio.astype(np.float64).mean(axis=1)
    # If the current array has an integer dtype, divide by that dtype's full-scale
    # range; otherwise convert to float64. The check occurs AFTER stereo averaging.
    if np.issubdtype(audio.dtype, np.integer):
        scale = max(abs(np.iinfo(audio.dtype).min), np.iinfo(audio.dtype).max)
        audio = audio.astype(np.float64) / scale
    else:
        audio = audio.astype(np.float64)

    # != means unequal. Resample only when rates differ; gcd finds their greatest
    # common divisor, and // gives integer factors for resample_poly(up, down).
    if sample_rate != target_sr:
        divisor = math.gcd(int(sample_rate), int(target_sr))
        audio = resample_poly(audio, target_sr // divisor, sample_rate // divisor)

    # Use the middle target_length samples, or pad shorter clips with zeros.
    # [start:stop] excludes stop; the returned waveform has 40,000 samples here.
    target_length = int(round(target_sr * seconds))
    if len(audio) >= target_length:
        start = (len(audio) - target_length) // 2
        audio = audio[start:start + target_length]
    else:
        left = (target_length - len(audio)) // 2
        right = target_length - len(audio) - left
        audio = np.pad(audio, (left, right))

    # This conditional expression chooses a peak only for a non-empty array.
    # Scale down only if its absolute peak exceeds 1; this is per-clip processing,
    # separate from training-derived feature standardisation later.
    peak = np.max(np.abs(audio)) if audio.size else 0.0
    if peak > 1.0:
        audio = audio / peak
    return audio.astype(np.float32)


# Convert hertz (Hz) to the perceptual mel scale; the following function
# reverses it. np.asarray lets these formulas accept scalars or array inputs.
def hz_to_mel(frequency_hz):
    return 2595.0 * np.log10(1.0 + np.asarray(frequency_hz) / 700.0)


def mel_to_hz(mel_value):
    return 700.0 * (10.0 ** (np.asarray(mel_value) / 2595.0) - 1.0)


# Create triangular frequency weights: 26 mel filters by 257 FFT bins.
# linspace chooses evenly spaced mel points; floor/astype find integer bins.
# clip restricts the bins to the available non-negative frequencies.
def mel_filter_bank(sample_rate=SAMPLE_RATE, n_fft=N_FFT, n_mels=N_MELS):
    mel_points = np.linspace(hz_to_mel(0), hz_to_mel(sample_rate / 2), n_mels + 2)
    hz_points = mel_to_hz(mel_points)
    bins = np.floor((n_fft + 1) * hz_points / sample_rate).astype(int)
    bins = np.clip(bins, 0, n_fft // 2)

    filters = np.zeros((n_mels, n_fft // 2 + 1), dtype=np.float64)
    # range excludes its upper bound. Each filter uses three adjacent bin points;
    # the two inner loops fill its rising and falling sides. shape[1] is its width.
    for index in range(1, n_mels + 1):
        left, centre, right = bins[index - 1:index + 2]
        centre = max(centre, left + 1)
        right = max(right, centre + 1)
        for frequency_bin in range(left, min(centre, filters.shape[1])):
            filters[index - 1, frequency_bin] = (frequency_bin - left) / (centre - left)
        for frequency_bin in range(centre, min(right, filters.shape[1])):
            filters[index - 1, frequency_bin] = (right - frequency_bin) / (right - centre)
    return filters


# DCT means discrete cosine transform: this returns a (20, 26) basis array.
# [:, None] adds a column dimension, allowing NumPy broadcasting to combine
# every coefficient index with every mel index without a Python inner loop.
def dct_basis(n_mfcc=N_MFCC, n_mels=N_MELS):
    mel_indices = np.arange(n_mels)
    coefficient_indices = np.arange(n_mfcc)[:, None]
    basis = np.cos(np.pi * coefficient_indices * (2 * mel_indices + 1) / (2 * n_mels))
    basis *= np.sqrt(2.0 / n_mels)
    basis[0] /= np.sqrt(2.0)
    return basis


# These are function calls: compute the two fixed bases once for all clips.
# Their definitions above describe how; the calls store their return values.
MEL_FILTERS = mel_filter_bank()
DCT_BASIS = dct_basis()


# Convert one mono waveform into (frames, 20) MFCCs. The pre-emphasis below
# subtracts 0.97 times the previous sample; [1:] and [:-1] align neighbouring samples.
def numpy_mfcc(audio):
    """Return frames x coefficients MFCCs using NumPy for the feature mathematics."""
    if audio.ndim != 1:
        raise ValueError("numpy_mfcc expects mono one-dimensional audio.")
    emphasized = np.empty_like(audio, dtype=np.float64)
    emphasized[0] = audio[0]
    emphasized[1:] = audio[1:] - 0.97 * audio[:-1]

    # Convert frame/hop durations to samples: 400 per frame and 160 between starts.
    # ceil rounds up so the last partial frame is included and zero-padded.
    frame_length = int(round(SAMPLE_RATE * FRAME_MS / 1000))
    hop_length = int(round(SAMPLE_RATE * HOP_MS / 1000))
    frame_count = 1 + int(np.ceil(max(0, len(emphasized) - frame_length) / hop_length))
    padded_length = (frame_count - 1) * hop_length + frame_length
    padded = np.pad(emphasized, (0, max(0, padded_length - len(emphasized))))
    # [None, :] is a row and [:, None] is a column. Broadcasting builds all frame
    # indices; padded[indices] gathers the samples into (249, 400) here. A Hamming
    # window tapers frame edges before the frequency transform.
    indices = (np.arange(frame_length)[None, :] +
               hop_length * np.arange(frame_count)[:, None])
    frames = padded[indices] * np.hamming(frame_length)

    # rfft transforms each frame along axis 1, padding 400 samples to 512 points.
    # Magnitude-squared gives power. @ means matrix multiplication; .T transposes
    # the filters so (249, 257) @ (257, 26) gives (249, 26) mel energies.
    spectrum = np.fft.rfft(frames, n=N_FFT, axis=1)
    power = (np.abs(spectrum) ** 2) / N_FFT
    mel_energy = np.maximum(power @ MEL_FILTERS.T, 1e-12)
    # The positive floor above prevents log(0). Log energies multiplied by the
    # transposed DCT basis give (249, 20); isfinite rejects NaN or infinite values.
    log_mel = np.log(mel_energy)
    mfcc = log_mel @ DCT_BASIS.T
    if not np.isfinite(mfcc).all():
        raise FloatingPointError("Non-finite MFCC value encountered.")
    return mfcc.astype(np.float32)


# For sequence/map alternatives, average each group of four frames and discard
# a leftover partial group. reshape(-1, factor, coefficients) infers the group
# count; mean(axis=1) averages within each group. 
def pool_mfcc_time(mfcc, factor=POOL_FACTOR):
    usable = (len(mfcc) // factor) * factor
    if usable == 0:
        raise ValueError("MFCC sequence is too short to pool.")
    return mfcc[:usable].reshape(-1, factor, mfcc.shape[1]).mean(axis=1)


# For aggregated features, gradient estimates frame-to-frame MFCC changes (deltas).
# Mean and standard deviation over time each give 20 values for MFCCs and
# deltas; concatenate joins these four vectors into 80 features per clip.
def aggregate_mfcc(mfcc):
    delta = np.gradient(mfcc.astype(np.float64), axis=0)
    vector = np.concatenate([
        mfcc.mean(axis=0), mfcc.std(axis=0),
        delta.mean(axis=0), delta.std(axis=0),
    ])
    if vector.shape != (80,):
        raise AssertionError(f"Expected 80 aggregated features, got {vector.shape}")
    return vector.astype(np.float32)


# Run WAV loading and MFCC extraction, then choose a return shape.
# "aggregated" returns (80,); the other branches return sequences or coefficient-by-time maps.
def extract_representation(path, representation):
    audio = load_wav_center_crop(path)
    mfcc = numpy_mfcc(audio)
    if representation == "aggregated":
        return aggregate_mfcc(mfcc)
    pooled = pool_mfcc_time(mfcc)
    if representation == "sequence":
        return pooled.astype(np.float32)
    # .T swaps time/coefficient axes; [None, :, :] adds a channel dimension.
    # The sequence/map outputs here would be (62, 20) and (1, 20, 62).
    if representation == "map":
        return pooled.T[None, :, :].astype(np.float32)  # channel x coefficients x time
    raise ValueError(f"Unknown representation: {representation}")

### Feature arrays, caches and scaling

Here `X` holds input features and `y` holds class labels. For `N` clips, aggregated `X.shape` is `(N, 80)` and `y.shape` is `(N,)`; rows must stay aligned with the corresponding file paths. An array's **shape** lists the length of each dimension. Axis `0` runs over clips; axis `1` runs over features.

The cache stores extracted features before standardisation. Scaling subtracts each **training** feature mean and divides by its training standard deviation plus a small constant. Validation uses those same values, so its feature distribution does not help fit the transformation. The feature means/stds remain in memory; this trial does not save them with a trained model.


In [5]:
# Hash the feature settings into a short cache-name tag. json.dumps turns
# the dictionary into text; sort_keys gives a consistent key order. This tag
# describes settings, not the contents of the audio files.
def feature_fingerprint(representation):
    parameters = {
        "sample_rate": SAMPLE_RATE,
        "clip_seconds": CLIP_SECONDS,
        "frame_ms": FRAME_MS,
        "hop_ms": HOP_MS,
        "n_fft": N_FFT,
        "n_mels": N_MELS,
        "n_mfcc": N_MFCC,
        "pool_factor": POOL_FACTOR,
        "representation": representation,
    }
    return hashlib.sha256(json.dumps(parameters, sort_keys=True).encode()).hexdigest()[:12]


# Select just the requested partition, then return (X, y, paths): feature
# rows, numeric labels and matching audio paths in the same row order.
def build_or_load_split_features(selected, split_name, representation):
    subset = selected[selected["split"] == split_name].reset_index(drop=True)
    fingerprint = feature_fingerprint(representation)
    cache_path = CACHE_DIR / f"{MODEL_TAG}_{RUN_MODE}_{split_name}_{representation}_{fingerprint}.npz"
    # Reuse a compressed NumPy .npz cache when present. np.load returns an
    # archive-like object indexed by saved names; allow_pickle=False avoids loading
    # pickled Python objects. Checks below compare row counts, labels and finiteness.
    if cache_path.exists():
        cached = np.load(cache_path, allow_pickle=False)
        X = cached["X"]
        y = cached["y"]
        paths = cached["paths"].astype(str)
        if len(X) != len(subset) or not np.array_equal(y, subset["label"].to_numpy()):
            raise ValueError(f"Cache does not match the current manifest: {cache_path}")
        if not np.isfinite(X).all():
            raise FloatingPointError(f"Cache contains non-finite values: {cache_path}")
        print(f"Loaded {split_name} {representation} cache: {X.shape}")
        return X, y, paths

    # If no cache exists, extract each clip in this partition. itertuples yields
    # row objects whose fields are accessed as attributes, such as row.path.
    features = []
    failures = []
    for row in subset.itertuples(index=False):
        try:
            features.append(extract_representation(row.path, representation))
        # try/except records extraction errors so the following RuntimeError reports
        # failures together. A failed clip is not silently removed from the labels.
        except Exception as exc:
            failures.append((row.path, repr(exc)))
    if failures:
        preview = "\n".join(f"- {path}: {error}" for path, error in failures[:10])
        raise RuntimeError(f"Feature extraction failed for {len(failures)} files:\n{preview}")

    # stack adds a sample axis: aggregated X has shape (N, 80), and y/paths have
    # shape (N,). save below writes unstandardised features, labels and paths.
    X = np.stack(features).astype(np.float32)
    y = subset["label"].to_numpy(dtype=np.int64)
    paths = subset["path"].astype(str).to_numpy(dtype=str)
    if not np.isfinite(X).all():
        raise FloatingPointError("Feature extraction produced non-finite values.")
    np.savez_compressed(cache_path, X=X, y=y, paths=paths)
    print(f"Saved {split_name} {representation} cache: {X.shape}")
    return X, y, paths
# Return three objects: the full file manifest, balanced split manifest and
# a dictionary containing only training and validation feature tuples.
def prepare_validation_features(representation):
    """Prepare train and validation features only for hyperparameter trials.

    The held-out test audio is deliberately not featurised in trial notebooks.
    Test evaluation is reserved for 07_MLP_Final_Selected_Model.ipynb.
    """
    manifest = scan_dataset()
    selected = make_balanced_splits(manifest)
    # This summary includes test file counts, but the calls below load/extract
    # audio features only for train and validation; no test features are requested.
    print(split_summary(selected).to_string(index=False))
    data = {
        "train": build_or_load_split_features(selected, "train", representation),
        "validation": build_or_load_split_features(selected, "validation", representation),
    }
    return manifest, selected, data


# Choose axes over which to compute one mean/std per feature or coefficient.
# For aggregated input (N, 80), axis 0 is the training-sample dimension.
def standardize_train_validation_features(X_train, X_validation, representation):
    if representation == "aggregated":
        axes = (0,)
    elif representation == "sequence":
        axes = (0, 1)
    elif representation == "map":
        axes = (0, 1, 3)  # one mean/std per MFCC coefficient
    else:
        raise ValueError(representation)
    # Fit scaling ONLY on training data. keepdims retains shape (1, 80), which
    # broadcasts across each row of either partition. Adding 1e-6 keeps the divisor
    # positive even for a constant training feature; validation never fits its own scale.
    mean = X_train.mean(axis=axes, keepdims=True)
    std = X_train.std(axis=axes, keepdims=True) + 1e-6
    # Return four arrays, in order: scaled train, scaled validation, training
    # mean and training std. The caller unpacks these into four variable names.
    return (
        ((X_train - mean) / std).astype(np.float32),
        ((X_validation - mean) / std).astype(np.float32),
        mean.astype(np.float32),
        std.astype(np.float32),
    )

### Reading the classification metrics

**Accuracy** is the fraction of labels predicted correctly. **Precision** asks how often predictions for a class are right; **recall** asks how many true members of that class are found. **F1** is their harmonic mean. **Macro** F1 averages the five class F1 scores equally, rather than weighting larger classes more. **Balanced accuracy** averages class recalls. This trial reports these on validation data, after restoring each model's best validation-loss epoch.


In [6]:
# These are scikit-learn evaluation functions; the neural network itself
# is implemented in NumPy. Predictions and matching labels go into the metrics.
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
)


# Inputs are matching true/predicted label arrays. Macro averages give each
# class equal weight; zero_division=0 returns zero for undefined scores.
# The unused fourth return value is assigned to _, a conventional placeholder.
def classification_metrics(y_true, y_pred):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=np.arange(len(CLASS_NAMES)),
        average="macro", zero_division=0,
    )
    # Return a metrics dictionary. Accuracy is the fraction correct; balanced
    # accuracy averages class recalls. float converts results to plain Python numbers.
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_precision": float(precision),
        "macro_recall": float(recall),
        "macro_f1": float(f1),
    }


# Defined for reuse but not called in this notebook. Rows are true classes,
# columns predicted classes; row-normalised colours show proportions, while
# cell text also shows raw counts. Return the counts and save/show the figure.
def plot_confusion(y_true, y_pred, title, output_path):
    matrix = confusion_matrix(y_true, y_pred, labels=np.arange(len(CLASS_NAMES)))
    row_sums = np.maximum(matrix.sum(axis=1, keepdims=True), 1)
    normalized = matrix / row_sums
    # subplots returns a Figure and an Axes object. Their methods add labels,
    # colours and text; nested loops visit every matrix cell. The conditional
    # text colour below is chosen from that cell's normalised value.
    fig, ax = plt.subplots(figsize=(8, 6))
    image = ax.imshow(normalized, cmap="Blues", vmin=0, vmax=1)
    for row in range(len(CLASS_NAMES)):
        for column in range(len(CLASS_NAMES)):
            ax.text(column, row, f"{matrix[row, column]}\n{normalized[row, column]:.2f}",
                    ha="center", va="center", fontsize=8,
                    color="white" if normalized[row, column] > 0.55 else "black")
    ax.set_xticks(range(len(CLASS_NAMES)), CLASS_NAMES, rotation=35, ha="right")
    ax.set_yticks(range(len(CLASS_NAMES)), CLASS_NAMES)
    ax.set_xlabel("Predicted generator")
    ax.set_ylabel("True generator")
    ax.set_title(title)
    fig.colorbar(image, ax=ax, label="Row-normalized proportion")
    fig.tight_layout()
    fig.savefig(output_path, dpi=160, bbox_inches="tight")
    plt.show()
    return matrix


# Serialise a Python payload to JSON (JavaScript Object Notation) text and
# write it to a Path. Existing files at the same path are overwritten.
def save_json(payload, path):
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print(f"Saved {path}")


# Another unused helper here: build a per-file prediction table and save CSV
# (comma-separated values). List comprehensions map label IDs to class names;
# index=False omits pandas row numbers. Return the table for possible reuse.
def save_prediction_table(paths, y_true, y_pred, output_path):
    table = pd.DataFrame({
        "path": paths,
        "true_label": y_true,
        "true_class": [CLASS_NAMES[index] for index in y_true],
        "predicted_label": y_pred,
        "predicted_class": [CLASS_NAMES[index] for index in y_pred],
    })
    table.to_csv(output_path, index=False)
    return table

## Steps

The assessed fully connected model implements every layer and optimizer directly with NumPy.

In [7]:
# This call actually runs the earlier data helpers. Unpack their three returns;
# then dictionary lookups retrieve each partition's (features, labels, paths).
manifest, selected, data = prepare_validation_features("aggregated")
X_train, y_train, train_paths = data["train"]
X_validation, y_validation, validation_paths = data["validation"]
# Replace the raw feature arrays with their standardised versions, using only
# training statistics for both. These statistics are retained in memory here.
X_train, X_validation, feature_mean, feature_std = standardize_train_validation_features(
    X_train, X_validation, "aggregated"
)
# Check feature width and that training contains all five class IDs. shape[0]
# is the sample count; shape[1] is feature count. Labels stay as integers.
assert X_train.shape[1] == 80
assert set(y_train) == set(range(len(CLASS_NAMES)))
print("Prepared validation-trial shapes:", X_train.shape, X_validation.shape)

### Reading the network code

A **class** describes a type of object. Creating `Dense(...)` makes one layer object with its own weights and biases. Its **attributes** store values (`layer.W`); its **methods** perform work (`layer.forward(...)`). A `def` statement defines a function/method; parentheses call it. `return` sends a result back to the caller.

The input has **80 features** and the output has **five classes**; `hidden_sizes` supplies the widths between them. Hidden layers apply their configured activation and dropout; softmax turns the final five scores into probabilities. A forward pass computes predictions. **Backpropagation** uses the chain rule to compute how loss changes with each parameter. The optimizer then uses those gradients to update the parameters; computing gradients and applying updates are separate steps.


In [8]:
# Input: one row of five raw scores (logits) per sample. Subtracting each row
# maximum avoids large exponentials; division returns probabilities summing
# to one along axis 1. No data partition is fixed by this reusable function.
def stable_softmax(logits):
    shifted = logits - np.max(logits, axis=1, keepdims=True)
    exponentials = np.exp(shifted)
    return exponentials / exponentials.sum(axis=1, keepdims=True)


# Map an incoming loss gradient through softmax. This general derivative is
# checked below; training uses the combined softmax/cross-entropy derivative.
def softmax_backward(upstream, probabilities):
    # Vector-Jacobian product for a standalone softmax layer.
    return probabilities * (upstream - np.sum(upstream * probabilities, axis=1, keepdims=True))


# Paired row/column indexing selects the true-class probability for each
# sample. Clip avoids log(0); the negative-log mean is one scalar loss.
# Confident correct predictions give a smaller value.
def cross_entropy(probabilities, labels):
    return float(-np.log(np.clip(probabilities[np.arange(len(labels)), labels], 1e-12, 1.0)).mean())


# A class defines an object type; Dense(...) creates one layer and calls
# __init__. self is that instance. self.W/self.b are stored attributes;
# forward/backward are methods called on it. rng is a random Generator object.
class Dense:
    def __init__(self, input_size, output_size, rng, initialization="he"):
        # He uses variance 2/input_size; the alternate Xavier setting uses 1/input_size.
        # Weights have (input_size, output_size); biases have (output_size,).
        scale = np.sqrt(2.0 / input_size) if initialization == "he" else np.sqrt(1.0 / input_size)
        self.W = (rng.standard_normal((input_size, output_size)) * scale).astype(np.float64)
        self.b = np.zeros(output_size, dtype=np.float64)

    def forward(self, inputs):
        # Store the (batch, input_size) inputs for backpropagation. Matrix multiply
        # then broadcast the bias to return (batch, output_size) scores.
        self.inputs = inputs
        return inputs @ self.W + self.b

    def backward(self, upstream):
        # upstream is the loss gradient from the next operation. Store gradients for
        # weights/biases, and return the gradient for the previous layer. Summing over
        # axis 0 combines sample contributions; the model has already divided by batch size.
        self.dW = self.inputs.T @ upstream
        self.db = upstream.sum(axis=0)
        return upstream @ self.W.T


# ReLU means rectified linear unit: keep positives, replace negatives with
# zero. Backward passes gradients only where the stored inputs were positive.
class ReLU:
    def forward(self, inputs):
        self.inputs = inputs
        return np.maximum(inputs, 0.0)

    def backward(self, upstream):
        return upstream * (self.inputs > 0)


# Sigmoid maps scores into (0, 1). Boolean masks split positive/negative
# inputs to avoid overflowing exp; ~positive selects the opposite positions.
# Backward uses the stored output in the derivative s * (1 - s).
class Sigmoid:
    def forward(self, inputs):
        self.output = np.empty_like(inputs, dtype=np.float64)
        positive = inputs >= 0
        self.output[positive] = 1.0 / (1.0 + np.exp(-inputs[positive]))
        exp_values = np.exp(inputs[~positive])
        self.output[~positive] = exp_values / (1.0 + exp_values)
        return self.output

    def backward(self, upstream):
        return upstream * self.output * (1.0 - self.output)


# During training, randomly remove activations with the given probability.
# Survivors are divided by the keep probability (inverted dropout); evaluation
# uses all activations unchanged. fixed_mask supports repeatable gradient checks.
class Dropout:
    def __init__(self, probability, rng):
        if not 0.0 <= probability < 1.0:
            raise ValueError("Dropout probability must be in [0, 1).")
        self.probability = probability
        self.rng = rng

    def forward(self, inputs, training, fixed_mask=None):
        if not training or self.probability == 0.0:
            self.mask = np.ones_like(inputs)
        elif fixed_mask is not None:
            self.mask = fixed_mask
        else:
            keep = self.rng.random(inputs.shape) >= self.probability
            self.mask = keep / (1.0 - self.probability)
        return inputs * self.mask

    # Reuse the forward mask so removed activations also receive zero gradient.
    def backward(self, upstream):
        return upstream * self.mask


# SGD = stochastic gradient descent: subtract the learning rate times
# each mini-batch gradient. The rate decreases with epoch when decay > 0;
# step returns that rate for the training history.
class SGDOptimizer:
    def __init__(self, learning_rate=0.01, decay=0.0):
        self.learning_rate = learning_rate
        self.decay = decay

    def step(self, layers, epoch):
        rate = self.learning_rate / (1.0 + self.decay * epoch)
        for layer in layers:
            layer.W -= rate * layer.dW
            layer.b -= rate * layer.db
        return rate


# Momentum keeps a velocity per layer, combining its previous update with
# the new gradient. id(layer) keys the stored arrays by object identity.
class MomentumOptimizer:
    def __init__(self, learning_rate=0.01, momentum=0.9, decay=0.0):
        self.learning_rate = learning_rate
        self.momentum = momentum
        self.decay = decay
        self.velocity = {}

    def step(self, layers, epoch):
        rate = self.learning_rate / (1.0 + self.decay * epoch)
        for layer in layers:
            key = id(layer)
            if key not in self.velocity:
                self.velocity[key] = [np.zeros_like(layer.W), np.zeros_like(layer.b)]
            velocity_w, velocity_b = self.velocity[key]
            # [:] updates the existing array in place, so the dictionary still refers
            # to the same velocity. += then applies that update to the layer parameters.
            velocity_w[:] = self.momentum * velocity_w - rate * layer.dW
            velocity_b[:] = self.momentum * velocity_b - rate * layer.db
            layer.W += velocity_w
            layer.b += velocity_b
        return rate


# MLP = multilayer perceptron: this object assembles the layers and handles
# forward prediction, loss, backpropagation and parameter snapshots.
class NumPyMLP:
    def __init__(self, input_size, hidden_sizes, hidden_activation, dropout_rate,
                 l1=0.0, l2=0.0, seed=RANDOM_SEED):
        self.rng = np.random.default_rng(seed)
        self.l1, self.l2 = l1, l2
        # *hidden_sizes expands the list of hidden widths between the input
        # width (80 for these features) and the five output classes. Adjacent widths
        # define the shape of each Dense layer; this also supports different depths.
        sizes = [input_size, *hidden_sizes, len(CLASS_NAMES)]
        self.dense_layers = []
        self.activations = []
        self.dropouts = []
        for index in range(len(sizes) - 1):
            initialization = "xavier" if hidden_activation == "sigmoid" else "he"
            self.dense_layers.append(Dense(sizes[index], sizes[index + 1], self.rng, initialization))
            # Only hidden layers receive an activation and dropout object. The final
            # dense layer outputs class logits; softmax is applied in forward instead.
            if index < len(sizes) - 2:
                self.activations.append(ReLU() if hidden_activation == "relu" else Sigmoid())
                self.dropouts.append(Dropout(dropout_rate, self.rng))

    def forward(self, inputs, training=False):
        output = inputs.astype(np.float64)
        # [:-1] excludes the final dense layer; zip walks corresponding objects
        # together. Read the nested calls inside out: dense, activation, then dropout.
        for dense, activation, dropout in zip(self.dense_layers[:-1], self.activations, self.dropouts):
            output = dropout.forward(activation.forward(dense.forward(output)), training=training)
        # [-1] selects the last layer. Its five logits become (batch, 5) probabilities.
        return stable_softmax(self.dense_layers[-1].forward(output))

    # Add optional L1 (absolute weight) and L2 (squared weight) penalties,
    # divided by this call's sample count. Biases are not penalised; strengths
    # come from the configuration, and zero disables the corresponding term.
    def loss(self, probabilities, labels):
        penalty = 0.0
        for layer in self.dense_layers:
            penalty += self.l1 * np.abs(layer.W).sum() + 0.5 * self.l2 * np.square(layer.W).sum()
        return cross_entropy(probabilities, labels) + penalty / len(labels)

    def backward(self, probabilities, labels):
        # For softmax with mean cross-entropy, start from probabilities minus a
        # one-hot target: subtract one at each true-label position, then divide by
        # batch size. copy prevents overwriting the forward probabilities.
        gradient = probabilities.copy()
        gradient[np.arange(len(labels)), labels] -= 1.0
        gradient /= len(labels)
        gradient = self.dense_layers[-1].backward(gradient)
        # Backpropagate in reverse layer order using the chain rule. These methods
        # store dW/db; they do not update weights until optimizer.step is called.
        for index in reversed(range(len(self.activations))):
            gradient = self.dropouts[index].backward(gradient)
            gradient = self.activations[index].backward(gradient)
            gradient = self.dense_layers[index].backward(gradient)
        # Add weight-penalty gradients for any nonzero L1/L2 setting.
        for layer in self.dense_layers:
            layer.dW += self.l1 * np.sign(layer.W) / len(labels) + self.l2 * layer.W / len(labels)

    # Evaluation mode disables dropout. argmax(axis=1) returns the largest-
    # probability class index for each sample, giving a one-dimensional label array.
    def predict(self, inputs):
        return self.forward(inputs, training=False).argmax(axis=1)

    # Copy weights/biases into a list of pairs, so later training cannot alter
    # the saved best state. load_state_dict restores those arrays in place.
    # These snapshots stay in memory; they do not create a model file.
    def state_dict(self):
        return [(layer.W.copy(), layer.b.copy()) for layer in self.dense_layers]

    def load_state_dict(self, state):
        for layer, (weights, bias) in zip(self.dense_layers, state):
            layer.W[:] = weights
            layer.b[:] = bias

    # .size counts scalar array entries; sum totals all weights and biases.
    def parameter_count(self):
        return int(sum(layer.W.size + layer.b.size for layer in self.dense_layers))

### Gradient checks

Finite differences verify dense, sigmoid, ReLU, softmax, and fixed-mask dropout derivatives.

The checks below use tiny synthetic arrays, not any coursework data partition. A **finite difference** nudges one input/weight slightly up and down and estimates the slope of a scalar objective. Comparing that slope with backpropagation helps check derivative calculations. A fixed dropout mask keeps the function unchanged between those measurements. These are sampled component checks, not proof that every aspect of training is correct.


In [9]:
# Perturb one array entry up/down, call objective each time, then restore it.
# The central difference estimates its derivative. objective is a function
# passed as an argument (without parentheses); objective() calls it.
def finite_difference(array, index, objective, epsilon=1e-5):
    original = array[index]
    array[index] = original + epsilon
    plus = objective()
    array[index] = original - epsilon
    minus = objective()
    array[index] = original
    return (plus - minus) / (2 * epsilon)


# Compare a backpropagated derivative with the numerical estimate, using a
# small denominator floor to avoid dividing by zero.
def relative_error(analytical, numerical):
    return abs(analytical - numerical) / max(1e-8, abs(analytical) + abs(numerical))


# These are small synthetic arrays, not train/validation/test audio. A toy
# 4-input, 5-hidden-unit MLP still has five output classes. forward/backward
# compute analytical gradients before selected weights are perturbed.
rng = np.random.default_rng(7)
check_x = rng.normal(size=(3, 4))
check_labels = np.array([0, 1, 2])
check_model = NumPyMLP(4, [5], "sigmoid", 0.0, seed=8)
check_probabilities = check_model.forward(check_x, training=True)
check_model.backward(check_probabilities, check_labels)

# This no-argument function reads the toy arrays/model from the surrounding
# namespace and returns a scalar loss for finite_difference to measure.
def check_objective():
    return cross_entropy(check_model.forward(check_x, training=False), check_labels)

checks = []
# Each tuple selects one weight by (row, column). Compare the stored dW
# entry with a numerical derivative, collecting relative errors in a list.
for index in [(0, 0), (2, 3), (3, 4)]:
    numerical = finite_difference(check_model.dense_layers[0].W, index, check_objective)
    analytical = check_model.dense_layers[0].dW[index]
    checks.append(relative_error(analytical, numerical))

# Standalone sigmoid and ReLU checks keep activation derivatives explicit.
sigmoid = Sigmoid()
sigmoid_input = np.array([[-1.2, 0.4], [1.1, -0.7]], dtype=float)
sigmoid_upstream = rng.normal(size=sigmoid_input.shape)
sigmoid.forward(sigmoid_input)
sigmoid_gradient = sigmoid.backward(sigmoid_upstream)
def sigmoid_objective():
    return float(np.sum(sigmoid.forward(sigmoid_input) * sigmoid_upstream))
# The scalar objective combines activation outputs with a chosen upstream
# gradient. This checks one input entry; ReLU repeats the same pattern below.
sigmoid_numeric = finite_difference(sigmoid_input, (0, 1), sigmoid_objective)
checks.append(relative_error(sigmoid_gradient[0, 1], sigmoid_numeric))

relu = ReLU()
relu_input = np.array([[-1.3, 0.7], [2.1, -0.9]], dtype=float)
relu_upstream = rng.normal(size=relu_input.shape)
relu.forward(relu_input)
relu_gradient = relu.backward(relu_upstream)
def relu_objective():
    return float(np.sum(relu.forward(relu_input) * relu_upstream))
relu_numeric = finite_difference(relu_input, (1, 0), relu_objective)
checks.append(relative_error(relu_gradient[1, 0], relu_numeric))

# Standalone softmax vector-Jacobian check.
# For softmax, compare one input-score derivative for two synthetic samples
# and five classes. This checks the standalone softmax_backward helper.
logits = rng.normal(size=(2, 5))
upstream = rng.normal(size=(2, 5))
probabilities = stable_softmax(logits)
softmax_gradient = softmax_backward(upstream, probabilities)
def softmax_objective():
    return float(np.sum(stable_softmax(logits) * upstream))
softmax_numeric = finite_difference(logits, (0, 2), softmax_objective)
checks.append(relative_error(softmax_gradient[0, 2], softmax_numeric))

# Fixed-mask inverted-dropout gradient check.
dropout = Dropout(0.5, rng)
dropout_input = rng.normal(size=(2, 3))
dropout_upstream = rng.normal(size=(2, 3))
# Fix the dropout mask so repeated objective evaluations represent the same
# operation; survivors have factor 2 because keep probability is 0.5.
fixed_mask = np.array([[2, 0, 2], [0, 2, 0]], dtype=float)
dropout.forward(dropout_input, training=True, fixed_mask=fixed_mask)
dropout_gradient = dropout.backward(dropout_upstream)
def dropout_objective():
    return float(np.sum(dropout_input * fixed_mask * dropout_upstream))
dropout_numeric = finite_difference(dropout_input, (0, 2), dropout_objective)
checks.append(relative_error(dropout_gradient[0, 2], dropout_numeric))

# Report the largest sampled relative error, then require it below 1e-5.
# These are spot checks of components, not exhaustive tests of every parameter.
print("Maximum MLP component gradient-check error:", max(checks))
assert max(checks) < 1e-5

### Learning rate, decay, and batch size experiment

This notebook changes one main modelling factor and saves validation outputs only. The held-out test split is reserved for the final selected-model notebook.

### Following this experiment

The learning rate controls update size; batch size controls how many training examples contribute to an update. The epoch-based rule is `initial_rate / (1 + decay * epoch)`. The list varies these settings in selected combinations; it is not an exhaustive grid.

An **epoch** visits all training rows in shuffled mini-batches. Only training batches update weights. Early stopping snapshots sufficiently improved validation loss and stops after the configured patience (default seven stale epochs), then restores the best snapshot. The results table separately ranks configurations by validation macro F1, then balanced accuracy and accuracy. Each candidate's history remains in memory, but only the winning history is written. `RUN_MODE` is fixed to `final`; these MLP trials do not use the analysis notebooks' smoke cap.


In [10]:
# Shuffle row indices, then yield matching feature/label batches. yield
# returns one pair at a time and resumes the loop when the caller asks for
# the next pair. A final batch can be smaller than batch_size.
def iterate_minibatches(X, y, batch_size, rng):
    order = rng.permutation(len(X))
    for start in range(0, len(X), batch_size):
        batch = order[start:start + batch_size]
        yield X[batch], y[batch]


# Take a settings dictionary; use the prepared global training/validation
# arrays. Return a trained MLP and a pandas history table, after restoring
# the epoch with the lowest validation loss.
def train_mlp(configuration):
    # Construct a fresh model from training feature width and configuration
    # values. get(key, default) supplies a fallback only when a key is absent;
    # seed=... is a keyword argument naming the receiving parameter.
    # Arguments before seed=... are matched to constructor parameters by
    # position. The keyword argument names its parameter explicitly; omitted
    # optional parameters would use the defaults in the definition.
    model = NumPyMLP(
        X_train.shape[1], configuration["hidden_sizes"], configuration["activation"],
        configuration["dropout"], configuration.get("l1", 0.0), configuration.get("l2", 0.0),
        seed=RANDOM_SEED,
    )
    # Choose the optimizer object: plain SGD for "sgd", momentum otherwise.
    # Momentum stores earlier updates with coefficient 0.9 in this implementation.
    if configuration["optimizer"] == "sgd":
        optimizer = SGDOptimizer(configuration["learning_rate"], configuration["decay"])
    else:
        optimizer = MomentumOptimizer(configuration["learning_rate"], 0.9, configuration["decay"])

    # An epoch is a pass through the training rows. get reads the requested
    # limit (default 50) and patience (default 7); min caps batch size at the
    # training-set size. Each configuration can therefore stop before its epoch limit.
    epochs = configuration.get("epochs", 50)
    patience = configuration.get("patience", 7)
    batch_size = min(configuration.get("batch_size", 64), len(X_train))
    rng = np.random.default_rng(RANDOM_SEED)
    history = []
    best_loss = np.inf
    best_state = None
    stale_epochs = 0

    # Only TRAINING mini-batches calculate gradients and update weights.
    # forward stores layer values; backward computes gradients; step applies them.
    # Dropout is active because training=True. The shuffle changes each epoch.
    for epoch in range(epochs):
        for batch_X, batch_y in iterate_minibatches(X_train, y_train, batch_size, rng):
            probabilities = model.forward(batch_X, training=True)
            model.backward(probabilities, batch_y)
            learning_rate = optimizer.step(model.dense_layers, epoch)

        # After updates, evaluate both whole partitions with dropout disabled.
        # These calls measure loss/accuracy; neither performs an optimizer update.
        # argmax(1) selects a class, == compares labels, and mean gives the correct fraction.
        train_probabilities = model.forward(X_train, training=False)
        validation_probabilities = model.forward(X_validation, training=False)
        row = {
            "epoch": epoch + 1,
            "learning_rate": learning_rate,
            "train_loss": model.loss(train_probabilities, y_train),
            "validation_loss": model.loss(validation_probabilities, y_validation),
            "train_accuracy": float((train_probabilities.argmax(1) == y_train).mean()),
            "validation_accuracy": float((validation_probabilities.argmax(1) == y_validation).mean()),
        }
        history.append(row)
        # Save a weight snapshot only if validation loss improves by more than
        # 1e-6. Otherwise count a stale epoch; break stops the epoch loop at patience.
        # Validation therefore influences stopping/selection, but supplies no gradients.
        if row["validation_loss"] < best_loss - 1e-6:
            best_loss = row["validation_loss"]
            best_state = model.state_dict()
            stale_epochs = 0
        else:
            stale_epochs += 1
            if stale_epochs >= patience:
                break

    # Restore the best saved epoch, rather than simply returning the last one.
    # The history still contains every epoch actually run, including later stale epochs.
    if best_state is None:
        best_state = model.state_dict()
    model.load_state_dict(best_state)
    return model, pd.DataFrame(history)


# Convert NumPy scalars/arrays into ordinary Python values that JSON accepts.
# isinstance checks an object's type; recursive calls handle nested dictionaries
# and lists. This helper returns converted data without writing a file.
def to_jsonable(value):
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, dict):
        return {key: to_jsonable(item) for key, item in value.items()}
    if isinstance(value, list):
        return [to_jsonable(item) for item in value]
    return value


# These configurations vary initial learning rate, batch size and/or
# epoch-based rate decay. They are listed candidates, not a Cartesian grid
# of every possible combination. decay changes the rate, not weight penalties.
trial_configs = [{'name': 'lr_0p003_batch_64',
  'hidden_sizes': [128, 64],
  'activation': 'relu',
  'dropout': 0.2,
  'l1': 0.0,
  'l2': 0.0,
  'optimizer': 'momentum',
  'learning_rate': 0.003,
  'decay': 0.01,
  'batch_size': 64,
  'epochs': 50},
 {'name': 'lr_0p006_batch_64',
  'hidden_sizes': [128, 64],
  'activation': 'relu',
  'dropout': 0.2,
  'l1': 0.0,
  'l2': 0.0,
  'optimizer': 'momentum',
  'learning_rate': 0.006,
  'decay': 0.01,
  'batch_size': 64,
  'epochs': 50},
 {'name': 'lr_0p01_batch_64',
  'hidden_sizes': [128, 64],
  'activation': 'relu',
  'dropout': 0.2,
  'l1': 0.0,
  'l2': 0.0,
  'optimizer': 'momentum',
  'learning_rate': 0.01,
  'decay': 0.01,
  'batch_size': 64,
  'epochs': 50},
 {'name': 'lr_0p02_batch_32',
  'hidden_sizes': [128, 64],
  'activation': 'relu',
  'dropout': 0.2,
  'l1': 0.0,
  'l2': 0.0,
  'optimizer': 'momentum',
  'learning_rate': 0.02,
  'decay': 0.01,
  'batch_size': 32,
  'epochs': 50},
 {'name': 'lr_0p01_batch_128',
  'hidden_sizes': [128, 64],
  'activation': 'relu',
  'dropout': 0.2,
  'l1': 0.0,
  'l2': 0.0,
  'optimizer': 'momentum',
  'learning_rate': 0.01,
  'decay': 0.01,
  'batch_size': 128,
  'epochs': 50},
 {'name': 'lr_0p01_decay_0p02',
  'hidden_sizes': [128, 64],
  'activation': 'relu',
  'dropout': 0.2,
  'l1': 0.0,
  'l2': 0.0,
  'optimizer': 'momentum',
  'learning_rate': 0.01,
  'decay': 0.02,
  'batch_size': 64,
  'epochs': 50}]

# Collect validation scores and per-epoch histories. perf_counter measures
# elapsed wall-clock time; the later duration also includes reporting work.
validation_rows, histories = [], {}
training_start = time.perf_counter()
# Each dictionary is one candidate. train_mlp starts fresh weights and
# optimizer state for it; the same training/validation split and seed are reused.
# Its returned model has restored the best validation-loss epoch.
for configuration in trial_configs:
    print(f"Training {configuration['name']}")
    model, history = train_mlp(configuration)
    # Score only VALIDATION data, using weights restored by train_mlp. The
    # training partition was used for fitting; test labels/audio are not evaluated.
    prediction = model.predict(X_validation)
    row = classification_metrics(y_validation, prediction)
    # update adds configuration details to the score dictionary. json.dumps stores
    # the hidden-width list as text within the CSV row for the comparison notebook.
    row.update({
        "trial": TRIAL_TAG,
        "trial_factor": "Learning rate, decay, and batch size",
        "variant": configuration["name"],
        "epochs_trained": int(len(history)),
        "parameter_count": model.parameter_count(),
        "hidden_sizes_json": json.dumps(configuration["hidden_sizes"]),
        "activation": configuration["activation"],
        "dropout": float(configuration.get("dropout", 0.0)),
        "l1": float(configuration.get("l1", 0.0)),
        "l2": float(configuration.get("l2", 0.0)),
        "optimizer": configuration["optimizer"],
        "learning_rate": float(configuration["learning_rate"]),
        "decay": float(configuration.get("decay", 0.0)),
        "batch_size": int(configuration.get("batch_size", 64)),
        "configured_epochs": int(configuration.get("epochs", 50)),
    })
    validation_rows.append(row)
    histories[configuration["name"]] = history

# Rank configurations by validation macro F1, then balanced accuracy,
# then accuracy, descending. This is selection across configurations; each
# configuration already restored its best epoch by validation LOSS.
# display is supplied by the notebook environment for readable tables.
validation_results = pd.DataFrame(validation_rows).sort_values(
    ["macro_f1", "balanced_accuracy", "accuracy"], ascending=False
).reset_index(drop=True)
display(validation_results)

# .iloc[0] selects the first ranked row; ["variant"] gets its name.
# Use that name to retrieve its history from the dictionary. Only this
# winning history is written below; the validation table includes all variants.
best_variant = validation_results.iloc[0]["variant"]
best_history = histories[best_variant]
validation_path = TABLES_DIR / f"validation_{TRIAL_TAG}_{RUN_MODE}.csv"
history_path = TABLES_DIR / f"training_history_{TRIAL_TAG}_{RUN_MODE}.csv"
summary_path = METRICS_DIR / f"trial_summary_{TRIAL_TAG}_{RUN_MODE}.json"
# Write validation scores and epoch history to CSV. RUN_MODE makes _final
# filenames here; it does not mean this notebook evaluates the final test set.
# Repeated runs write to the same paths and overwrite those files.
validation_results.to_csv(validation_path, index=False)
best_history.to_csv(history_path, index=False)

# Two Axes objects show loss and accuracy against epoch for training and
# validation. axes[0]/axes[1] choose a panel; plot uses history-table columns.
# A train/validation gap can help interpret how well the fitted model generalises.
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(best_history["epoch"], best_history["train_loss"], label="train")
axes[0].plot(best_history["epoch"], best_history["validation_loss"], label="validation")
axes[0].set(title="Loss", xlabel="Epoch", ylabel="Cross-entropy")
axes[1].plot(best_history["epoch"], best_history["train_accuracy"], label="train")
axes[1].plot(best_history["epoch"], best_history["validation_accuracy"], label="validation")
axes[1].set(title="Accuracy", xlabel="Epoch", ylabel="Accuracy")
# Apply the same grid/legend methods to both panels; savefig writes a PNG
# image and show displays it. Neither call changes model parameters.
for axis in axes:
    axis.grid(alpha=0.2)
    axis.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / f"training_curves_{TRIAL_TAG}_{RUN_MODE}.png", dpi=160)
plt.show()

# Assemble the JSON summary: validation-only selection, settings, scores,
# elapsed time and paths to the CSV files. No test metrics or model weights
# are saved by this trial; feature caches were handled earlier.
payload = {
    "model": "NumPy MLP",
    "trial": TRIAL_TAG,
    "trial_factor": "Learning rate, decay, and batch size",
    "run_mode": RUN_MODE,
    "selection_metric": "validation_macro_f1_then_balanced_accuracy_then_accuracy",
    "selection_scope": "validation_only",
    "test_evaluation": "not_performed_in_this_notebook",
    "best_variant": best_variant,
    "best_validation": validation_results.iloc[0].to_dict(),
    "configs": trial_configs,
    "runtime_seconds": float(time.perf_counter() - training_start),
    "validation_table": str(validation_path),
    "best_history_table": str(history_path),
}
# Read nested calls inside out: convert the payload first, then save its
# JSON text using the earlier file-writing helper.
save_json(to_jsonable(payload), summary_path)
print(f"Saved validation outputs to {OUTPUT_DIR}")
